# repo-code-completion: GPU demo (Ollama + qwen2.5-coder:7b)

Runs the indexer -> BM25/Symbol/Dependency retrieval -> LLM selection pipeline against a GPU-accelerated local Ollama server, instead of CPU-only.

**Before running**: Runtime -> Change runtime type -> select a GPU (T4 is fine).

## 1. Install and start Ollama, pull the model

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!ollama pull qwen2.5-coder:7b

In [ ]:
# Confirm it's up and (ideally) running on GPU, not CPU.
!curl -s http://localhost:11434/api/tags | head -c 300
print()
!ollama ps

## 2. Get the project code onto Colab

Two options -- use whichever applies:

- **Option A (if you've pushed this repo to GitHub)**: set `REPO_URL` below and run the cell.
- **Option B (no GitHub remote yet)**: zip your local `repo-code-completion` folder, then run the *next* cell instead -- it will prompt you to upload the zip.

In [ ]:
# Option A: clone from GitHub (skip this cell if you're using Option B).
REPO_URL = "https://github.com/Robertkiza0/repo-code.git"
if REPO_URL:
    !git clone "$REPO_URL" repo-code-completion

In [ ]:
# Option B: upload a zip of your local repo-code-completion folder.
import os, zipfile

if not os.path.exists("repo-code-completion"):
    from google.colab import files
    print("Zip your local repo-code-completion folder, then upload it here.")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(".")
    # if the zip's top-level entry isn't already named repo-code-completion,
    # find the extracted folder and rename it to match.
    if not os.path.exists("repo-code-completion"):
        extracted = [n.split("/")[0] for n in zipfile.ZipFile(zip_name).namelist()]
        top = sorted(set(extracted))[0]
        os.rename(top, "repo-code-completion")

In [ ]:
%cd repo-code-completion

## 3. Install only what this pipeline needs

(Skips the heavy generation-stage deps in `requirements.txt` like `torch`/`vllm` -- not needed for indexing/retrieval/selection.)

In [ ]:
!pip install -q tree-sitter tree-sitter-python tree-sitter-java tree-sitter-typescript tree-sitter-c-sharp rank-bm25 requests

## 4. Run the pipeline: index -> retrieve -> select

In [ ]:
from indexer.repo_parser import RepoParser
from retrieval.bm25_retriever import BM25Retriever
from retrieval.symbol_retriever import SymbolRetriever
from retrieval.dependency_retriever import DependencyRetriever
from retrieval.candidate_pipeline import CandidatePipeline
from selection.llm_selector import LLMSelector

chunks = [c.to_dict() for c in RepoParser("tests/sample_repo").parse_repo()]
pipeline = CandidatePipeline(BM25Retriever(chunks), SymbolRetriever(chunks), DependencyRetriever(chunks))

code_before_cursor = "result = Greeter("
target_file = "pkg/module_b.py"

candidates = pipeline.nominate(code_before_cursor, target_file=target_file)
print(f"{len(candidates)} candidates:")
for c in candidates:
    print(f"  {c['name']:12s} sources={c['sources']} scores={c['scores']}")

In [ ]:
import time

selector = LLMSelector(chunks)  # defaults to localhost:11434, qwen2.5-coder:7b
t0 = time.time()
result = selector.select(code_before_cursor, target_file, candidates)
print(f"took {time.time() - t0:.1f}s (should be seconds on GPU, not minutes)")
print()
print("selected_chunk_ids:         ", result["selected_chunk_ids"])
print("candidate_chunk_ids:        ", result["candidate_chunk_ids"])
print("rejected_hallucinated_ids:  ", result["rejected_hallucinated_ids"])